# 01 Exploratory data analysis: credit card customer segmentation

**Business question:** Which groups of credit card customers exist based on how they spend, pay, and borrow, and what does each group look like in business terms?

**Data:** the CC GENERAL dataset. It has about 8,950 rows (one per credit card holder) and 18 columns, with behaviour measured over the last 6 months. We found it as `CC_GENERAL.csv` and copied it to `data/raw/credit_card.csv`.

**How to read this notebook:** every code cell has a short note above it with three lines (what it does, what to look for, what the output means). Imports sit inside the cells that first use them, with a comment saying why.

## Step 1: Early EDA with ydata-profiling

*EDA* means exploratory data analysis: looking at the data before modelling so nothing surprises us later.

**What this cell does:** Finds the project folder and loads the CSV into a table (a pandas DataFrame).  
**What to look for:** That the shape matches what we expect: about 8,950 rows and 18 columns.  
**What the output means:** `shape` is (rows, columns). If it were far off, we would have loaded the wrong file.

In [1]:
# Imported here so this step only loads the library when it runs
from pathlib import Path
import pandas as pd


def find_project_root(start: Path = Path.cwd()) -> Path:
    """Walk up from the current folder until we see pyproject.toml (nbconvert runs inside notebooks/)."""
    return next(folder for folder in [start, *start.parents] if (folder / "pyproject.toml").exists())


ROOT = find_project_root()
DATA_PATH = ROOT / "data" / "raw" / "credit_card.csv"
REPORTS_DIR = ROOT / "reports"

df = pd.read_csv(DATA_PATH)  # read the CSV into a DataFrame
print(f"{df.shape=}")
df.head()  # first 5 rows, to eyeball the columns

df.shape=(8950, 18)


,CUST_ID,BALANCE,BALANCE_FREQUENCY,PURCHASES,ONEOFF_PURCHASES,INSTALLMENTS_PURCHASES,CASH_ADVANCE,PURCHASES_FREQUENCY,ONEOFF_PURCHASES_FREQUENCY,PURCHASES_INSTALLMENTS_FREQUENCY,CASH_ADVANCE_FREQUENCY,CASH_ADVANCE_TRX,PURCHASES_TRX,CREDIT_LIMIT,PAYMENTS,MINIMUM_PAYMENTS,PRC_FULL_PAYMENT,TENURE
0,C10001,40.900749,0.818182,95.40,0.00,95.4,0.000000,0.166667,0.000000,0.083333,0.000000,0,2,1000.0,201.802084,139.509787,0.000000,12
1,C10002,3202.467416,0.909091,0.00,0.00,0.0,6442.945483,0.000000,0.000000,0.000000,0.250000,4,0,7000.0,4103.032597,1072.340217,0.222222,12
2,C10003,2495.148862,1.000000,773.17,773.17,0.0,0.000000,1.000000,1.000000,0.000000,0.000000,0,12,7500.0,622.066742,627.284787,0.000000,12
3,C10004,1666.670542,0.636364,1499.00,1499.00,0.0,205.788017,0.083333,0.083333,0.000000,0.083333,1,1,7500.0,0.000000,NaN,0.000000,12
4,C10005,817.714335,1.000000,16.00,16.00,0.0,0.000000,0.083333,0.083333,0.000000,0.000000,0,1,1200.0,678.334763,244.791237,0.000000,12


**What this cell does:** Lists each column's data type (dtype) and counts missing values and duplicate rows.  
**What to look for:** Any column that is not numeric (we expect only CUST_ID), the columns with missing values, and the duplicate count.  
**What the output means:** Numeric dtypes mean we can compute on them directly. Missing values must be handled before clustering because KMeans cannot accept blanks. Duplicated rows would count one customer twice.

In [2]:
print(f"{df.dtypes.value_counts().to_dict()=}")  # how many columns of each type
print(f"{df.duplicated().sum()=}")  # fully identical rows
print(f"{df['CUST_ID'].is_unique=}")  # every customer ID should appear exactly once

# One-liner: count blanks per column, keep only columns that have some, show count and percent.
# isna() marks blanks as True, sum() counts them, and the mask [> 0] drops complete columns.
missing = df.isna().sum()
missing_table = pd.DataFrame({"missing": missing, "percent": (missing / len(df) * 100).round(2)})[missing > 0]
missing_table

df.dtypes.value_counts().to_dict()={dtype('float64'): 14, dtype('int64'): 3, dtype('O'): 1}
df.duplicated().sum()=np.int64(0)
df['CUST_ID'].is_unique=True


,missing,percent
CREDIT_LIMIT,1,0.01
MINIMUM_PAYMENTS,313,3.50


**What this cell does:** Prints summary statistics (count, mean, spread, min, quartiles, max) for the numeric columns.  
**What to look for:** Big gaps between the median (50%) and the max, and columns that are mostly zero.  
**What the output means:** When the max is far above the 75% value, a few customers have extreme values (a long right tail). That matters later because KMeans is pulled by extremes.

In [3]:
numeric_columns = df.drop(columns="CUST_ID").columns.tolist()  # every column except the ID
df[numeric_columns].describe().T.round(2)  # .T flips the table so each feature is one row

,count,mean,std,min,25%,50%,75%,max
BALANCE,8950.0,1564.47,2081.53,0.00,128.28,873.39,2054.14,19043.14
BALANCE_FREQUENCY,8950.0,0.88,0.24,0.00,0.89,1.00,1.00,1.00
PURCHASES,8950.0,1003.20,2136.63,0.00,39.64,361.28,1110.13,49039.57
ONEOFF_PURCHASES,8950.0,592.44,1659.89,0.00,0.00,38.00,577.40,40761.25
INSTALLMENTS_PURCHASES,8950.0,411.07,904.34,0.00,0.00,89.00,468.64,22500.00
CASH_ADVANCE,8950.0,978.87,2097.16,0.00,0.00,0.00,1113.82,47137.21
PURCHASES_FREQUENCY,8950.0,0.49,0.40,0.00,0.08,0.50,0.92,1.00
ONEOFF_PURCHASES_FREQUENCY,8950.0,0.20,0.30,0.00,0.00,0.08,0.30,1.00
PURCHASES_INSTALLMENTS_FREQUENCY,8950.0,0.36,0.40,0.00,0.00,0.17,0.75,1.00
CASH_ADVANCE_FREQUENCY,8950.0,0.14,0.20,0.00,0.00,0.00,0.22,1.50


**What this cell does:** Builds the full ydata-profiling HTML report and saves it in reports/.  
**What to look for:** Open reports/credit_card_profile.html in a browser. Look at the Alerts tab, the Correlations tab, and each variable's histogram.  
**What the output means:** The report automatically flags skew, many zeros, high correlation and missing values. It is a fast first look, not a final answer.

In [4]:
def build_profile(frame: pd.DataFrame, output_path: Path):
    """Create the profiling report and save it as HTML."""
    # Imported here so the notebook opens fast and this step only loads the library when it runs
    from ydata_profiling import ProfileReport

    report = ProfileReport(frame, title="Credit card customers: profile", minimal=False, progress_bar=False)
    report.to_file(output_path)  # writes the interactive HTML page
    return report


REPORTS_DIR.mkdir(exist_ok=True)
profile = build_profile(df.drop(columns="CUST_ID"), REPORTS_DIR / "credit_card_profile.html")  # ID is not a feature
print(f"{(REPORTS_DIR / 'credit_card_profile.html').exists()=}")

/tmp/ipykernel_658/232935934.py:4: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


  0%|          | 0/17 [00:00<?, ?it/s]

 71%|███████   | 12/17 [00:00<00:00, 119.56it/s]

100%|██████████| 17/17 [00:00<00:00, 123.94it/s]

(REPORTS_DIR / 'credit_card_profile.html').exists()=True


**What this cell does:** Prints the warnings (alerts) that ydata-profiling raised, so we can quote them in our findings.  
**What to look for:** Alert types such as SKEWED, ZEROS, HIGH_CORRELATION, MISSING.  
**What the output means:** Each alert is a data-quality hint. We read them here so our written findings come from real output, not guesses.

In [5]:
alerts = profile.get_description().alerts  # list of Alert objects
alert_table = pd.DataFrame({"column": [a.column_name for a in alerts], "alert": [a.alert_type.name for a in alerts]})
print(alert_table.groupby("alert")["column"].apply(lambda cols: ", ".join(sorted(cols))).to_string())

alert
HIGH_CORRELATION    BALANCE, BALANCE_FREQUENCY, CASH_ADVANCE, CASH...
MISSING                                              MINIMUM_PAYMENTS
ZEROS               CASH_ADVANCE, CASH_ADVANCE_FREQUENCY, CASH_ADV...


**What this cell does:** Checks three structural facts the report may hint at: whether PURCHASES equals one-off plus installment purchases, how many rows have zero purchases, and how many have MINIMUM_PAYMENTS above PAYMENTS.  
**What to look for:** The share of rows where the purchase identity holds, and the counts of zero-purchase and odd payment rows.  
**What the output means:** If PURCHASES is just the sum of two other columns, it adds no new information (redundancy). Customers with zero purchases are a natural group (cash-advance or inactive users).

In [6]:
# Does PURCHASES equal ONEOFF + INSTALLMENTS? np.isclose allows tiny rounding differences.
import numpy as np  # imported here because this is the first cell that needs numpy

identity_holds = np.isclose(df["PURCHASES"], df["ONEOFF_PURCHASES"] + df["INSTALLMENTS_PURCHASES"], atol=0.01)
print(f"{identity_holds.mean()=:.4f}")
print(f"{(df['PURCHASES'] == 0).sum()=}")
print(f"{(df['MINIMUM_PAYMENTS'] > df['PAYMENTS']).sum()=}")
print(f"{df['TENURE'].value_counts().sort_index().to_dict()=}")
print(f"{df['CREDIT_LIMIT'].isna().sum()=}, {df['MINIMUM_PAYMENTS'].isna().sum()=}")

identity_holds.mean()=0.9979
(df['PURCHASES'] == 0).sum()=np.int64(2044)
(df['MINIMUM_PAYMENTS'] > df['PAYMENTS']).sum()=np.int64(2365)
df['TENURE'].value_counts().sort_index().to_dict()={6: 204, 7: 190, 8: 196, 9: 175, 10: 236, 11: 365, 12: 7584}
df['CREDIT_LIMIT'].isna().sum()=np.int64(1), df['MINIMUM_PAYMENTS'].isna().sum()=np.int64(313)


### Five findings from the profile

- **The table is clean in structure.** There are 8,950 rows, no duplicate rows, and every `CUST_ID` is unique. *Why it matters:* one row is one customer, so no de-duplication is needed and cluster sizes can be read directly as shares of customers.
- **Missing values are few and limited to two columns.** `MINIMUM_PAYMENTS` is missing for 313 customers (3.50%) and `CREDIT_LIMIT` for just 1. *Why it matters:* KMeans cannot accept blanks, but this is small enough to fill in with a simple rule inside a pipeline (Step 4) instead of throwing customers away.
- **Money columns are extremely right-skewed.** Skewness (how lopsided a distribution is; 0 is symmetric) is 13.6 for `MINIMUM_PAYMENTS`, 10.0 for `ONEOFF_PURCHASES` and 8.1 for `PURCHASES`. The top `PURCHASES` value is about 49,000 while 99% of customers are below about 9,000. *Why it matters:* KMeans measures straight-line distance, so a few huge values would drag the centres and hide the smaller groups. We will transform the skewed columns (Step 5).
- **Many columns are mostly zeros.** 2,044 customers (about 23%) have made no purchases at all, and the cash advance columns are about 52% zeros. *Why it matters:* these zeros are real behaviour, not errors. They hint at natural groups (for example customers who only take cash advances), and they mean we should use `log1p` (log of 1 + x), which is safe at zero, rather than a plain log.
- **Several features say nearly the same thing.** `PURCHASES` equals `ONEOFF_PURCHASES` plus `INSTALLMENTS_PURCHASES` in 99.8% of rows, and pairs such as `CASH_ADVANCE_FREQUENCY` and `CASH_ADVANCE_TRX` move together almost perfectly (Spearman correlation 0.98). *Why it matters:* duplicated information counts twice in a distance calculation, so we will decide which features to drop in Step 3.

**Other things noticed:** 2,365 customers have a minimum payment larger than their payments (we will look at this in Step 4), and 7,584 of 8,950 customers (85%) have `TENURE` of 12 months, so tenure carries little information to separate groups.